In [12]:
import os
import joblib
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

In [13]:
DATA_PATH = "../dataset/UNSW_NB15_testing-set.parquet"

dataset = pd.read_parquet(DATA_PATH)

print(dataset.shape)
dataset.head()

(82332, 36)


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.09375,180363632.0,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.00000,881000000.0,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.00000,854400000.0,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.65625,600000000.0,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.00000,850400000.0,...,0,0,2,1,0,0,0,0,Normal,0


In [14]:
LIVE_FEATURES = [
    "dur",
    "proto",
    "spkts",
    "dpkts",
    "sbytes",
    "dbytes",
    "rate",
    "sload",
    "dload",
    "sinpkt",
    "dinpkt",
    "smean",
    "dmean"
]

X = dataset[LIVE_FEATURES].copy()
y = dataset["label"].copy()

print(X.shape)
print(y.value_counts())

(82332, 13)
label
1    45332
0    37000
Name: count, dtype: int64


In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (65865, 13)
Testing: (16467, 13)


In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "proto_encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["proto"]
        )
    ],
    remainder="passthrough"
)

In [17]:
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", rf)
])

In [18]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [19]:
param_grid = {

    "rf__n_estimators": [
        100,
        200,
        300
    ],

    "rf__max_depth": [
        10,
        20,
        30,
        None
    ],

    "rf__min_samples_split": [
        2,
        5,
        10
    ],

    "rf__min_samples_leaf": [
        1,
        2,
        4
    ],

    "rf__max_features": [
        "sqrt",
        "log2"
    ],

    "rf__class_weight": [
        None,
        "balanced"
    ]
}

In [20]:
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=2,
    return_train_score=True
)

In [21]:
grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'rf__class_weight': [None, 'balanced'], 'rf__max_depth': [10, 20, ...], 'rf__max_features': ['sqrt', 'log2'], 'rf__min_samples_leaf': [1, 2, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"return_train_score return_train_score: bool, default=FalseIf ``False``, the ``cv_results_`` attribute will not include trainingscores.Computing training scores is used to get insights on how differentparameter settings impact the overfitting/underfitting trade-off.However computing the scores on the training set can be computationallyexpensive and is not strictly required to select the parameters thatyield the best generalization performance... versionadded:: 0.19.. versionchanged:: 0.21 Default value was changed from ``True`` to ``False``",True
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, t

In [27]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

best_model = grid_search.best_estimator_

y_pred = best_model.predict(X_test)

print("TEST RESULTS")
print("-------------------------")
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

TEST RESULTS
-------------------------
Accuracy : 0.9340499180178539
Precision: 0.9608499826769835
Recall   : 0.9176133230395941
F1 Score : 0.9387340629583663

Confusion Matrix:
[[7061  339]
 [ 747 8320]]


In [22]:
print(
    "Best CV F1:",
    grid_search.best_score_
)

print("\nBest Parameters:")

for parameter, value in grid_search.best_params_.items():
    print(parameter, ":", value)

Best CV F1: 0.9409222828047923

Best Parameters:
rf__class_weight : balanced
rf__max_depth : None
rf__max_features : sqrt
rf__min_samples_leaf : 1
rf__min_samples_split : 5
rf__n_estimators : 100


In [23]:
best_pipeline = grid_search.best_estimator_

y_pred = best_pipeline.predict(
    X_test
)

print(
    "Accuracy:",
    accuracy_score(y_test, y_pred)
)

print(
    "Precision:",
    precision_score(y_test, y_pred)
)

print(
    "Recall:",
    recall_score(y_test, y_pred)
)

print(
    "F1:",
    f1_score(y_test, y_pred)
)

print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)

Accuracy: 0.9340499180178539
Precision: 0.9608499826769835
Recall: 0.9176133230395941
F1: 0.9387340629583663

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.95      0.93      7400
           1       0.96      0.92      0.94      9067

    accuracy                           0.93     16467
   macro avg       0.93      0.94      0.93     16467
weighted avg       0.94      0.93      0.93     16467


Confusion Matrix:
[[7061  339]
 [ 747 8320]]


In [24]:
best_preprocessor = (
    best_pipeline.named_steps[
        "preprocessor"
    ]
)

best_rf = (
    best_pipeline.named_steps[
        "rf"
    ]
)

In [25]:
TUNED_DIR = "../models/tuned"

os.makedirs(
    TUNED_DIR,
    exist_ok=True
)

joblib.dump(
    best_rf,
    os.path.join(
        TUNED_DIR,
        "tuned_random_forest.pkl"
    )
)

joblib.dump(
    best_preprocessor,
    os.path.join(
        TUNED_DIR,
        "tuned_preprocessor.pkl"
    )
)

print(
    "Tuned model saved successfully"
)

Tuned model saved successfully


In [26]:
results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1 Score"
    ],

    "Old RF": [
        0.9336,
        0.9555,
        0.9224,
        0.9387
    ],

    "Tuned RF": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred)
    ]
})

results

,Metric,Old RF,Tuned RF
0,Accuracy,0.9336,0.934050
1,Precision,0.9555,0.960850
2,Recall,0.9224,0.917613
3,F1 Score,0.9387,0.938734


In [28]:
import os
import joblib

os.makedirs("../models/tuned", exist_ok=True)

joblib.dump(
    best_model,
    "../models/tuned/tuned_rf_pipeline.pkl"
)

print("Tuned model saved successfully")

Tuned model saved successfully
